# Staging Area

## ORDERS

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE VIEW second_warehouse.staging.orders_view AS
SELECT order_id, customer_id, order_at AS order_date, region, total_usd AS order_cost
FROM second_warehouse.raw.raw_orders

In [ ]:
%%sql -r dataframe_2
SELECT * FROM second_warehouse.staging.orders_view

In [ ]:
%%sql -r dataframe_3
SELECT DISTINCT REGION FROM second_warehouse.staging.orders_view

In [ ]:
%%sql -r dataframe_4
UPDATE second_warehouse.raw.raw_orders SET REGION = 'EMEA' WHERE REGION = 'europe';

In [ ]:
CREATE OR REPLACE VIEW second_warehouse.staging.orders_clean_view AS
SELECT 
    order_id, 
    customer_id, 
    order_at AS order_date, 
    CASE WHEN region = 'europe' THEN 'EMEA' ELSE region END AS region,
    total_usd AS order_cost
FROM raw.raw_orders
QUALIFY ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY order_at DESC) = 1;

## Shipments

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE VIEW second_warehouse.staging.shipments_view AS
SELECT * FROM second_warehouse.raw.raw_shipments;

In [ ]:
%%sql -r dataframe_7
SELECT * FROM second_warehouse.staging.shipments_view

In [ ]:
%%sql -r dataframe_8
SELECT shipment_id, order_id, carrier,
    shipped_at::DATE AS shipping_date,
    delivered_at::DATE AS delivery_date,
    shipping_cost
FROM second_warehouse.staging.shipments_view;

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE VIEW second_warehouse.staging.shipments_clean_view AS (
SELECT *,
    CASE
        WHEN delivery_date IS NULL THEN 'Not Delivered'
        ELSE 'Delivered' END AS Delivery_status
FROM 
    (SELECT shipment_id, order_id, carrier,
    shipped_at::DATE AS shipping_date,
    delivered_at::DATE AS delivery_date,
    shipping_cost
FROM second_warehouse.staging.shipments_view));

# Mart

In [ ]:
CREATE OR REPLACE TABLE second_warehouse.marts.fct_fulfillment AS
WITH orders AS (
    SELECT * FROM second_warehouse.staging.orders_clean_view
),
shipments AS (
    SELECT * FROM second_warehouse.staging.shipments_clean_view
),
final_calculations AS (
    SELECT 
        o.order_id,
        o.customer_id,
        o.region,
        o.order_cost,
        s.shipping_cost,
        s.carrier,
        s.shipping_date,
        s.delivery_date,
        -- All calculations in one place
        DATEDIFF('day', o.order_date, s.shipping_date) AS lead_time_shipping,
        DATEDIFF('day', s.shipping_date, s.delivery_date) AS lead_time_delivery,
        (o.order_cost - s.shipping_cost) AS net_margin_usd
    FROM orders o
    LEFT JOIN shipments s USING(order_id)
)
SELECT 
    *,
    -- Logic built on top of the calculations above
    CASE
        WHEN delivery_date IS NULL AND lead_time_shipping >= 2 THEN 'LOST_OR_DELAYED'
        WHEN lead_time_delivery <= 1 THEN 'FAST_TRACK'
        ELSE 'STANDARD' 
    END AS order_status
FROM final_calculations;